# 00 — Data Exploration

**Project:** E-Commerce User Behavior Analysis & Recommendation System  
**Notebook purpose:** Initial exploration of the raw dataset — understanding structure, 
data types, missing values, and data quality issues before any cleaning is done.

## Environment Setup

This notebook was run on **Kaggle Notebooks** using the dataset attached directly 
from the Kaggle dataset page. No local setup or downloads are required to run it.

### To reproduce this notebook

1. Go to the dataset page on Kaggle:  
   [E-Commerce Behavior Data from Multi-Category Store](https://www.kaggle.com/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store)
2. Click the three dots then **New Notebook** to open a fresh Kaggle notebook with the dataset already attached
3. Click 'File' -> 'Import Notebook' and select the  `notebooks/00_data_exploration.ipynb` file from the repo
4. Run all cells top to bottom

Make sure that the datasets are loaded into the notebook and check their sizes.

In [1]:
import os

# Data set file path
data_path = '/kaggle/input/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store'

# List every file in that folder and size
for file in os.listdir(data_path):
    size_b = os.path.getsize(f'{data_path}/{file}') / (1024 * 1024 * 1024)
    print(f'{file}  —  {size_b:.1f} GB')

2019-Nov.csv  —  8.4 GB
2019-Oct.csv  —  5.3 GB


## Examine the first few rows

In [2]:
import pandas as pd

data_path = '/kaggle/input/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store'

# Load the first 5 rows of the dataset
df_peek = pd.read_csv(f'{data_path}/2019-Oct.csv', nrows=5)

print(df_peek.shape)   # how many rows and columns
print(df_peek)         # show the actual rows

(5, 9)
                event_time event_type  product_id          category_id  \
0  2019-10-01 00:00:00 UTC       view    44600062  2103807459595387724   
1  2019-10-01 00:00:00 UTC       view     3900821  2053013552326770905   
2  2019-10-01 00:00:01 UTC       view    17200506  2053013559792632471   
3  2019-10-01 00:00:01 UTC       view     1307067  2053013558920217191   
4  2019-10-01 00:00:04 UTC       view     1004237  2053013555631882655   

                         category_code     brand    price    user_id  \
0                                  NaN  shiseido    35.79  541312140   
1  appliances.environment.water_heater      aqua    33.20  554748717   
2           furniture.living_room.sofa       NaN   543.10  519107250   
3                   computers.notebook    lenovo   251.74  550050854   
4               electronics.smartphone     apple  1081.98  535871217   

                           user_session  
0  72d76fde-8bb3-4e00-8c23-a032dfed738c  
1  9333dfbd-b87a-4708-9857-6336

### First few rows observations
From the output above, we can create loose definitions for each column:
* event_time: when an action happened
* event_type: what the user did
* product_id: which product was involved
* category_id: a long number representing the product category
* category_code: possibly a human-readable version of the category
* brand: the product brand
* price: the product price in dollars
* user_id: which user performed the action
* user_session: a unique ID for the browsing session.

There are some things that stand out and may have to address later:
* category_id is not human-readable
* category_code has a NaN value
* brand also has a NaN value

## Examine a larger sample (missing values, data types, unique values)
Look at a larger sample so we can get a better understanding:

In [3]:
df_sample = pd.read_csv(f'{data_path}/2019-Oct.csv', nrows=100_000)

print('Shape:', df_sample.shape,'\n')

print('--- Missing values per column ---')
print(df_sample.isnull().sum(),'\n')

Shape: (100000, 9) 

--- Missing values per column ---
event_time           0
event_type           0
product_id           0
category_id          0
category_code    32587
brand            14393
price                0
user_id              0
user_session         0
dtype: int64 



### Missing Values
* category_code is missing about 32,500 row (roughly 1 in 3 are missing out of the sample)
* brand is missing about 14,400 rows (roughly 1 in 7)
* All other rows are fully populated

Thinking about the domain, the missing values may be normal in this kind of dataset, as some products may not be categorized or branded in a retail system.

In [4]:
print('--- Data types ---')
print(df_sample.dtypes,'\n')

--- Data types ---
event_time        object
event_type        object
product_id         int64
category_id        int64
category_code     object
brand             object
price            float64
user_id            int64
user_session      object
dtype: object 



### Data Types
* event_time has an 'object' datatype, meaning that it is treated as plain text, not a real date. We will need to convert this later
* All other data types seem reasonable (prices are floats, IDs are integers)

In [5]:
print('--- Unique values in event_type ---')
print(df_sample['event_type'].value_counts(),'\n')

--- Unique values in event_type ---
event_type
view        97130
purchase     1655
cart         1215
Name: count, dtype: int64 



### Event Types
Out of 100,000 events:
* about 97,000 are views (97%)
* about 1,200 are cart additions (1.6%)
* about 1,600 are purchases (1.2%)

This makes sense for an e-commerce website, as most people browse without buying. This also means that when we analyze conversion rates later, the numbers will be small percentages.

## Look at duplicate rows, price statistics, and rows with missing values

In [6]:
print('--- Duplicate rows ---')
print(df_sample.duplicated().sum(), '\n')

--- Duplicate rows ---
17 



### Duplicates
* There are 17 duplicate rows in 100,000
* A duplicate means the exact same user did the exact same thing to the exact same product at the exact same time, which shouldn't be possible.
* Although it's a small percentage, we must remove them because it is bad data

In [7]:
print('--- Price statistics ---')
print(df_sample['price'].describe(), '\n')

--- Price statistics ---
count    100000.000000
mean        286.827664
std         360.304691
min           0.000000
25%          61.010000
50%         154.380000
75%         357.110000
max        2574.070000
Name: price, dtype: float64 



### Price Statistics
* The minimum price is $0.00
* This doesn't make sense for a real purchase. This may be an error or an indication of free items.
* We will want to flag them later when we do data cleaning/processing.
* From the range, we can see that this online store has a variety of prices.
* Or, this may be an indicator that it did have a price, but was marked incorrectly
* All other statistics seem reasonable.

In [8]:
print('--- Sample of rows where category_code is missing ---')
print(df_sample[df_sample['category_code'].isnull()].head(3))

--- Sample of rows where category_code is missing ---
                event_time event_type  product_id          category_id  \
0  2019-10-01 00:00:00 UTC       view    44600062  2103807459595387724   
6  2019-10-01 00:00:08 UTC       view    17300353  2053013553853497655   
7  2019-10-01 00:00:08 UTC       view    31500053  2053013558031024687   

  category_code     brand   price    user_id  \
0           NaN  shiseido   35.79  541312140   
6           NaN     creed  380.96  555447699   
7           NaN  luminarc   41.16  550978835   

                           user_session  
0  72d76fde-8bb3-4e00-8c23-a032dfed738c  
6  4fe811e9-91de-46da-90c3-bbd87ed3a65d  
7  6280d577-25c8-4147-99a7-abc6048498d6  


### Missing category_code rows
* The rows do not seem to have an other problems besides the missing category_code
* This may mean that the category code was simply not recorded
* We won't delete these rows. Instead we will just fill in the missing category_code with some value ("unknown"). 

# Summary of Observations
* event_time is text that needs to be converted to datetime
* category_code has ~32% missing values. We need to fill with "unknown"
* we need to remove duplicate rows
* Some prices are zero so we either remove the rows or replace the price with 'NaN'. 